# Evaluate saved answers with Nemotron 120B and DeepEval

Run the cells below in order. **No GPU or model reload is needed.**

This evaluates134 already-saved answers (67 untuned and67 V1) using3 fixed criteria. It does not train a model or regenerate annotations. The judge uses your NVIDIA API key and receives expression evidence, saved outputs, and anonymized expert content for the reference criterion.

Put your key in Colab Secrets as **NVIDIA_API_KEY**, then enable notebook access. Do not paste it into saved code. Upload **deepeval-nemotron120b-v1.zip** when prompted. Results save on Drive and are resumable. Report ratings as exploratory automated assessment, separate from expert validation.

This version streams subprocess output and prints a saved-score count every15seconds. It uses the same data ZIP, rubric, and results folder. Leave any currently running evaluation alone; use this version for a later start or resume.


In [ ]:
import os, sys, subprocess
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "deepeval==4.3.3", "openai==3.46.0"])
print("Dependencies ready. No model was loaded.")


In [ ]:
from google.colab import files, drive
from pathlib import Path
import zipfile, json, hashlib

drive.mount("/content/drive")
UPLOAD_DIR = Path("/content/gtc_deepeval_upload")
UPLOAD_DIR.mkdir(exist_ok=True)
print("Upload deepeval-nemotron120b-v1.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith(".zip")]
assert len(zip_names) == 1, "Upload exactly the evaluation package ZIP."
archive = UPLOAD_DIR / Path(zip_names[0]).name
archive.write_bytes(uploaded[zip_names[0]])
with zipfile.ZipFile(archive) as z:
    for member in z.namelist():
        assert (UPLOAD_DIR / member).resolve().is_relative_to(UPLOAD_DIR.resolve()), "Unsafe ZIP path"
    z.extractall(UPLOAD_DIR)
PACKAGE = UPLOAD_DIR / "deepeval-nemotron120b-v1"
for name, expected in json.loads((PACKAGE / "inventory.json").read_text()).items():
    assert hashlib.sha256((PACKAGE / name).read_bytes()).hexdigest() == expected, name
OUTPUT = Path("/content/drive/MyDrive/GTC2027/SCPCP000004/nemotron120b_deepeval_v1")
print("Package verified: 134 saved answers, 3 criteria, 402 score requests.")
print("Judge:", json.loads((PACKAGE / "protocol.json").read_text())["judge"]["model"])
print("Results folder:", OUTPUT)


In [ ]:
# Key stays in memory and is passed privately to the evaluation subprocess.
from getpass import getpass
API_KEY = os.environ.get("NVIDIA_API_KEY")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("NVIDIA_API_KEY")
    except Exception:
        API_KEY = None
if not API_KEY:
    API_KEY = getpass("NVIDIA API key (hidden): ")
assert API_KEY, "An NVIDIA API key is required."
child_env = os.environ.copy()
child_env["NVIDIA_API_KEY"] = API_KEY
child_env["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"
import threading, time, signal

def run_with_live_progress(command, env, output_folder, heartbeat_seconds=15):
    print("Starting the judge. Progress will appear below.", flush=True)
    process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               start_new_session=True)
    def relay():
        for line in process.stdout:
            print(line, end="", flush=True)
    relay_thread = threading.Thread(target=relay, daemon=True)
    relay_thread.start()
    started = time.monotonic()
    try:
        while True:
            try:
                returncode = process.wait(timeout=heartbeat_seconds)
                break
            except subprocess.TimeoutExpired:
                saved = Path(output_folder) / "scores.jsonl"
                complete = failed = 0
                if saved.exists():
                    with saved.open() as handle:
                        for line in handle:
                            try:
                                row = json.loads(line)
                            except json.JSONDecodeError:
                                continue  # The last line may still be being written.
                            complete += row.get("status") == "complete"
                            failed += row.get("status") == "failed"
                elapsed = int(time.monotonic() - started)
                print(f"Progress: {complete}/402 complete, {failed} failed; "
                      f"{elapsed}s elapsed in this session. Waiting for active requests.", flush=True)
    except KeyboardInterrupt:
        print("Stopping this subprocess before resuming. Saved scores remain on Drive.", flush=True)
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise
    finally:
        relay_thread.join(timeout=2)
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

# Use this only after the earlier execution cell has finished.
# Do not start a second run while the original subprocess is still active.
run_with_live_progress(
    [sys.executable, "-u", str(PACKAGE / "run_evaluation.py"),
     "--package", str(PACKAGE), "--output", str(OUTPUT), "--workers", "2"],
    env=child_env, output_folder=OUTPUT)


In [ ]:
from IPython.display import display, Markdown
summary = json.loads((OUTPUT / "summary.json").read_text())
print("Complete:", summary["complete"], "/402; failed:", summary["failed"], "; pending:", summary["pending"])
display(Markdown((OUTPUT / "REPORT.md").read_text()))
result_zip = OUTPUT.parent / (OUTPUT.name + ".zip")
print("Return this results ZIP for analysis:", result_zip.name)
files.download(str(result_zip))
